# Kaggle Textbook Content Generator

This notebook processes any `TEXTBOOKS` folder sequentially, chapter by chapter. It prefers Gemma 4 E4B for multimodal image+text reasoning.

It will:
- extract text with Docling when available, otherwise fall back to PyMuPDF
- save chapterwise source artifacts
- extract and save images
- detect NCERT-style experiments/activities
- generate summary, key points, concepts, glossary, flashcards, misconceptions, applications, and at least 20 MCQs per chapter
- resume safely after Kaggle interruptions using per-chapter manifests, caches, and prior output datasets mounted under `/kaggle/input`

In [ ]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate", "bitsandbytes", "sentencepiece", "pymupdf", "pillow"])
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "docling"])
except Exception:
    pass


In [ ]:
import hashlib
import json
import logging
import os
import re
import shutil
import zipfile
from collections import Counter
from datetime import datetime
from pathlib import Path
from typing import Any

import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

try:
    from docling.document_converter import DocumentConverter
    DOCLING_AVAILABLE = True
except Exception:
    DOCLING_AVAILABLE = False

try:
    import fitz
    PYMUPDF_AVAILABLE = True
except Exception:
    PYMUPDF_AVAILABLE = False

TEXTBOOKS_ROOT = Path(os.environ.get('TEXTBOOKS_ROOT', '/kaggle/input/TEXTBOOKS'))
if not TEXTBOOKS_ROOT.exists():
    roots = [p for p in Path('/kaggle/input').iterdir() if p.is_dir()]
    pdf_roots = []
    for root in roots:
        if any(root.rglob('*.pdf')):
            pdf_roots.append(root)
    if pdf_roots:
        TEXTBOOKS_ROOT = pdf_roots[0]

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

OUTPUT_ROOT = Path(os.environ.get('OUTPUT_ROOT', '/kaggle/working/textbook_artifacts'))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT = OUTPUT_ROOT / '_cache'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
GENERATE_VOICE_PACK = os.environ.get('GENERATE_VOICE_PACK', '1') == '1'
VOICE_CHUNK_CHARS = int(os.environ.get('VOICE_CHUNK_CHARS', '1200'))
VOICE_MAX_ITEMS = int(os.environ.get('VOICE_MAX_ITEMS', '25'))

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_LOG_PATH = OUTPUT_ROOT / f'run_{RUN_ID}.log'
DEFAULT_GPU_MEMORY_GIB = int(os.environ.get('GPU_MEMORY_GIB', '12'))
DEFAULT_CPU_OFFLOAD_GIB = os.environ.get('CPU_OFFLOAD_GIB', '32GiB')
LOAD_IN_4BIT = os.environ.get('LOAD_IN_4BIT', '1') == '1'
PROMPT_IMAGE_COUNT = int(os.environ.get('PROMPT_IMAGE_COUNT', '1'))
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True

MODEL_CANDIDATES = [
    os.environ.get('MODEL_ID', 'google/gemma-4-e4b-it'),
    'google/gemma-4-e4b-it',
    'Qwen/Qwen2.5-7B-Instruct',
    'Qwen/Qwen2.5-3B-Instruct',
    'microsoft/Phi-3.5-mini-instruct',
]
MAX_SOURCE_CHARS = int(os.environ.get('MAX_SOURCE_CHARS', '8000'))
MAX_CHAPTERS = None
ONLY_SLUGS = None  # example: ['number_systems']
QUIZ_COUNT = 20
FLASHCARD_COUNT = 15
CONCEPT_COUNT = 10
GLOSSARY_COUNT = 12
KEY_POINT_COUNT = 5
MISCONCEPTION_COUNT = 8
APPLICATION_COUNT = 8
GRADE_MIN = int(os.environ.get('GRADE_MIN', '6'))
GRADE_MAX = int(os.environ.get('GRADE_MAX', '10'))


def configure_logging() -> logging.Logger:
    logger = logging.getLogger('textbook_generator')
    logger.setLevel(logging.INFO)
    logger.propagate = False
    if not logger.handlers:
        formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
        stream_handler = logging.StreamHandler()
        stream_handler.setFormatter(formatter)
        file_handler = logging.FileHandler(RUN_LOG_PATH, encoding='utf-8')
        file_handler.setFormatter(formatter)
        logger.addHandler(stream_handler)
        logger.addHandler(file_handler)
    return logger


LOGGER = configure_logging()


def gpu_summary() -> str:
    if not torch.cuda.is_available():
        return 'cpu'
    parts = []
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        parts.append(f'cuda:{index} {props.name} {props.total_memory / 1024**3:.1f}GiB')
    return ' | '.join(parts)


def build_max_memory() -> dict[Any, str] | None:
    if not torch.cuda.is_available():
        return None
    memory = {index: f'{DEFAULT_GPU_MEMORY_GIB}GiB' for index in range(torch.cuda.device_count())}
    memory['cpu'] = DEFAULT_CPU_OFFLOAD_GIB
    return memory


def should_process_grade(pdf_path: Path) -> bool:
    grade = infer_grade(pdf_path)
    return grade is not None and GRADE_MIN <= grade <= GRADE_MAX


LOGGER.info('TEXTBOOKS_ROOT=%s', TEXTBOOKS_ROOT)
LOGGER.info('OUTPUT_ROOT=%s', OUTPUT_ROOT)
LOGGER.info('DOCLING_AVAILABLE=%s', DOCLING_AVAILABLE)
LOGGER.info('PYMUPDF_AVAILABLE=%s', PYMUPDF_AVAILABLE)
LOGGER.info('GPU_COUNT=%s', torch.cuda.device_count() if torch.cuda.is_available() else 0)
LOGGER.info('GPU_STATE=%s', gpu_summary())
LOGGER.info('LOAD_IN_4BIT=%s | DEFAULT_GPU_MEMORY_GIB=%s | CPU_OFFLOAD=%s', LOAD_IN_4BIT, DEFAULT_GPU_MEMORY_GIB, DEFAULT_CPU_OFFLOAD_GIB)


In [ ]:
def normalize_space(text: str) -> str:
    return re.sub(r'\s+', ' ', str(text or '').replace('\u00a0', ' ')).strip()

def slugify(text: str) -> str:
    text = normalize_space(text).lower()
    text = re.sub(r'[^a-z0-9]+', '_', text)
    return re.sub(r'_+', '_', text).strip('_')

def title_case(text: str) -> str:
    text = normalize_space(text).replace('_', ' ')
    text = re.sub(r'\b_s\b', "'s", text)
    return text.title().replace("'S", "'s")

def file_hash(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def chapter_slug_from_pdf(pdf_path: Path) -> str:
    return slugify(pdf_path.stem)

def infer_grade(pdf_path: Path) -> int | None:
    for part in pdf_path.parts:
        m = re.search(r'(?:class|grade)\s*[_-]?(\d{1,2})', part.lower())
        if m:
            return int(m.group(1))
    return None

def infer_subject(pdf_path: Path) -> str:
    parts = [p.lower() for p in pdf_path.parts]
    if any('mathematics' in p or 'math' == p or p.startswith('math ') for p in parts):
        return 'mathematics'
    if any('science' in p for p in parts):
        return 'science'
    if any('social' in p or 'evs' in p for p in parts):
        return 'social_science'
    return slugify(pdf_path.parent.name or 'textbook')

def discover_pdfs(root: Path) -> list[Path]:
    return sorted([p for p in root.rglob('*.pdf') if p.is_file()], key=lambda p: str(p).lower())

def is_in_textbook_corpus(pdf_path: Path) -> bool:
    lower = str(pdf_path).lower()
    return 'textbooks' in lower or 'class ' in lower or 'class_' in lower or 'grade_' in lower

def output_dir_for(pdf_path: Path) -> Path:
    grade = infer_grade(pdf_path)
    subject = infer_subject(pdf_path)
    chapter = chapter_slug_from_pdf(pdf_path)
    grade_dir = f'grade_{grade}' if grade is not None else 'grade_unknown'
    return OUTPUT_ROOT / grade_dir / subject / chapter


RESUME_INPUT_ROOT = Path(os.environ.get('RESUME_INPUT_ROOT', '/kaggle/input'))
RESUME_SOURCE_HINTS = [Path(item.strip()) for item in os.environ.get('RESUME_SOURCE_HINTS', '').split(',') if item.strip()]
RESUME_SUBDIR_NAMES = [item.strip() for item in os.environ.get('RESUME_SUBDIR_NAMES', 'textbook_artifacts').split(',') if item.strip()]


def read_json_if_exists(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return None


def resume_roots() -> list[Path]:
    roots: list[Path] = []
    if RESUME_INPUT_ROOT.exists():
        for path in RESUME_INPUT_ROOT.iterdir():
            if path.is_dir():
                roots.append(path)
            elif path.is_file() and path.suffix.lower() == '.zip' and zipfile.is_zipfile(path):
                extracted_root = CACHE_ROOT / '_resume_import' / path.stem
                marker = extracted_root / '.extract_complete'
                if not marker.exists():
                    extracted_root.mkdir(parents=True, exist_ok=True)
                    shutil.unpack_archive(str(path), str(extracted_root), 'zip')
                    marker.write_text(path.name, encoding='utf-8')
                roots.append(extracted_root)
    for hint in RESUME_SOURCE_HINTS:
        if hint.exists() and hint.is_dir():
            roots.append(hint)
    unique_roots: list[Path] = []
    seen: set[str] = set()
    for root in roots:
        key = str(root)
        if key not in seen:
            seen.add(key)
            unique_roots.append(root)
    return unique_roots


def chapter_resume_candidates(chapter_dir: Path) -> list[Path]:
    rel = chapter_dir.relative_to(OUTPUT_ROOT)
    candidates: list[Path] = []
    for root in resume_roots():
        candidate = root / rel
        if candidate.is_dir():
            candidates.append(candidate)
        for subdir_name in RESUME_SUBDIR_NAMES:
            nested = root / subdir_name / rel
            if nested.is_dir():
                candidates.append(nested)
    return candidates


def chapter_artifacts_match_pdf(chapter_dir: Path, pdf_hash: str) -> bool:
    manifest = read_json_if_exists(chapter_dir / 'manifest.json')
    if manifest and manifest.get('pdf_hash') == pdf_hash:
        return True
    source_payload = read_json_if_exists(chapter_dir / 'source' / 'chapter_source.json')
    return bool(source_payload and source_payload.get('pdf_hash') == pdf_hash)


def import_resume_chapter(chapter_dir: Path, resume_dir: Path) -> None:
    chapter_dir.mkdir(parents=True, exist_ok=True)
    shutil.copytree(resume_dir, chapter_dir, dirs_exist_ok=True)


def iter_resume_chapter_dirs() -> list[Path]:
    chapter_dirs: list[Path] = []
    seen: set[str] = set()
    search_roots = resume_roots()
    for root in search_roots:
        for marker_name in ('manifest.json', 'chapter_source.json'):
            for marker in root.rglob(marker_name):
                chapter_dir = marker.parent if marker.name == 'manifest.json' else marker.parent.parent
                key = str(chapter_dir)
                if chapter_dir.exists() and chapter_dir.is_dir() and key not in seen:
                    seen.add(key)
                    chapter_dirs.append(chapter_dir)
    return chapter_dirs


def locate_resume_chapter(chapter_dir: Path, pdf_hash: str) -> Path | None:
    for candidate in chapter_resume_candidates(chapter_dir):
        if chapter_artifacts_match_pdf(candidate, pdf_hash):
            return candidate
    for candidate in iter_resume_chapter_dirs():
        if chapter_artifacts_match_pdf(candidate, pdf_hash):
            return candidate
    return None


In [ ]:
def dump_model(value: Any) -> dict[str, Any]:
    if hasattr(value, 'export_to_dict'):
        return value.export_to_dict()
    if hasattr(value, 'model_dump'):
        return value.model_dump(mode='json')
    if isinstance(value, dict):
        return value
    return {}

def extract_with_docling(pdf_path: Path) -> dict[str, Any]:
    converter = DocumentConverter()
    result = converter.convert(str(pdf_path))
    document = dump_model(result.document)
    texts = document.get('texts') if isinstance(document.get('texts'), list) else []
    tables = document.get('tables') if isinstance(document.get('tables'), list) else []
    pictures = document.get('pictures') if isinstance(document.get('pictures'), list) else []
    parts = []
    for item in texts:
        if not isinstance(item, dict):
            continue
        text = item.get('text') or item.get('orig') or item.get('content')
        if isinstance(text, str) and text.strip():
            parts.append(normalize_space(text))
    section_text = '\n\n'.join(parts)
    return {'document': document, 'section_text': section_text, 'tables': tables, 'pictures': pictures}

def extract_with_pymupdf(pdf_path: Path) -> dict[str, Any]:
    if not PYMUPDF_AVAILABLE:
        raise RuntimeError('PyMuPDF not available')
    parts = []
    images = []
    figures_dir = None
    with fitz.open(pdf_path) as pdf:
        for page_index, page in enumerate(pdf, start=1):
            text = normalize_space(page.get_text('text'))
            if text:
                parts.append(text)
            for img_index, img in enumerate(page.get_images(full=True), start=1):
                xref = img[0]
                base = pdf.extract_image(xref)
                if not base:
                    continue
                ext = base.get('ext', 'png')
                data = base.get('image')
                if not data:
                    continue
                images.append({'page': page_index, 'index': img_index, 'ext': ext, 'bytes': data, 'xref': xref})
    return {'section_text': '\n\n'.join(parts), 'images': images, 'tables': [], 'pictures': []}

def extract_chapter_source(pdf_path: Path) -> dict[str, Any]:
    payload = {'extractor': 'pymupdf', 'title': title_case(pdf_path.stem), 'section_text': '', 'images': [], 'tables': [], 'pictures': []}
    if DOCLING_AVAILABLE:
        try:
            doc = extract_with_docling(pdf_path)
            payload.update(doc)
            payload['extractor'] = 'docling'
            payload['title'] = title_case(pdf_path.stem)
            if len(payload.get('section_text', '')) > 100:
                return payload
        except Exception:
            pass
    if PYMUPDF_AVAILABLE:
        fallback = extract_with_pymupdf(pdf_path)
        payload.update(fallback)
    return payload

def split_sections_from_text(text: str) -> list[dict[str, Any]]:
    lines = [normalize_space(line) for line in str(text or '').splitlines()]
    sections = []
    current = {'title': 'Start', 'content': []}
    for line in lines:
        if not line:
            continue
        is_heading = (
            len(line) <= 80 and (
                re.match(r'^chapter\s+\d+', line, re.I)
                or re.match(r'^\d+(?:\.\d+)*\s+', line)
                or line.isupper()
                or line.lower().startswith(('experiment', 'activity', 'exercise', 'what is', 'why', 'how'))
            )
        )
        if is_heading:
            if current['content']:
                sections.append({'title': current['title'], 'text': ' '.join(current['content'])})
            current = {'title': line, 'content': []}
        else:
            current['content'].append(line)
    if current['content']:
        sections.append({'title': current['title'], 'text': ' '.join(current['content'])})
    return sections

def detect_experiments(sections: list[dict[str, Any]], full_text: str) -> list[dict[str, Any]]:
    experiments = []
    keywords = ('experiment', 'activity', 'investigation', 'try this', 'do this', 'observe', 'procedure')
    for section in sections:
        title = section['title']
        text = section['text']
        blob = f"{title} {text}".lower()
        if any(k in blob for k in keywords):
            experiments.append({'title': title, 'text': text[:2500], 'kind': 'experiment_or_activity'})
    if not experiments:
        for match in re.finditer(r'(?im)^(experiment|activity|investigation|try this|do this)\b.*$', full_text):
            experiments.append({'title': normalize_space(match.group(0))[:180], 'text': normalize_space(match.group(0))[:2500], 'kind': 'experiment_or_activity'})
    dedup = []
    seen = set()
    for item in experiments:
        key = item['title'].lower()
        if key not in seen:
            dedup.append(item)
            seen.add(key)
    return dedup[:20]

def extract_formulas(text: str) -> list[str]:
    pattern = re.compile(r'([A-Za-z][A-Za-z0-9_ ()]{0,40}\s*(?:=|∝|≤|≥|<|>)\s*[^.;\n]{1,100}|[A-Za-z0-9]+\s*/\s*[A-Za-z0-9]+)')
    items = []
    for match in pattern.finditer(text):
        formula = normalize_space(match.group(0))
        if 4 <= len(formula) <= 120 and formula not in items:
            items.append(formula)
    return items[:20]

def save_image_assets(pdf_path: Path, chapter_dir: Path, extracted: dict[str, Any]) -> list[dict[str, Any]]:
    image_dir = chapter_dir / 'images'
    image_dir.mkdir(parents=True, exist_ok=True)
    manifest = []

    def render_visible_image(pdf: Any, page_index: int, xref: int, out: Path) -> bool:
        try:
            page = pdf[page_index - 1]
            rects = page.get_image_rects(xref)
            if not rects:
                return False
            rect = max(rects, key=lambda rect: rect.width * rect.height)
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2), clip=rect, alpha=False)
            pix.save(str(out))
            return True
        except Exception:
            return False

    images = extracted.get('images') or []
    if images and isinstance(images, list) and PYMUPDF_AVAILABLE:
        with fitz.open(pdf_path) as pdf:
            for idx, item in enumerate(images, start=1):
                page_index = int(item.get('page') or 0)
                xref = int(item.get('xref') or 0)
                out = image_dir / f'image_{idx:03d}.png'
                rendered = False
                if page_index > 0 and xref > 0:
                    rendered = render_visible_image(pdf, page_index, xref, out)
                if not rendered:
                    data = item.get('bytes')
                    if not data:
                        continue
                    ext = item.get('ext', 'png')
                    out = image_dir / f'image_{idx:03d}.{ext}'
                    out.write_bytes(data)
                manifest.append({'file': str(out.relative_to(chapter_dir)), 'page': item.get('page'), 'index': item.get('index'), 'xref': item.get('xref')})
    elif PYMUPDF_AVAILABLE:
        with fitz.open(pdf_path) as pdf:
            for page_index, page in enumerate(pdf, start=1):
                for img_index, img in enumerate(page.get_images(full=True), start=1):
                    xref = img[0]
                    out = image_dir / f'page_{page_index:03d}_{img_index:02d}.png'
                    if render_visible_image(pdf, page_index, xref, out):
                        manifest.append({'file': str(out.relative_to(chapter_dir)), 'page': page_index, 'index': img_index, 'xref': xref})
                        continue
                    base = pdf.extract_image(xref)
                    if not base:
                        continue
                    data = base.get('image')
                    if not data:
                        continue
                    ext = base.get('ext', 'png')
                    out = image_dir / f'page_{page_index:03d}_{img_index:02d}.{ext}'
                    out.write_bytes(data)
                    manifest.append({'file': str(out.relative_to(chapter_dir)), 'page': page_index, 'index': img_index, 'xref': xref})
    return manifest


def source_excerpt(text: str, limit: int = 5000) -> str:
    return trim(text, limit)


def notes_to_text(notes: dict[str, Any]) -> str:
    return json.dumps(notes, ensure_ascii=False, indent=2)


def select_useful_images(chapter_dir: Path, source_text: str, experiments: list[dict[str, Any]], max_images: int = 1) -> list[Image.Image]:
    images = load_chapter_images(chapter_dir, max_images=4)
    if not images:
        return []
    text = source_text.lower()
    visual_markers = ('figure', 'diagram', 'graph', 'table', 'chart', 'map', 'illustration', 'experiment', 'activity', 'procedure')
    if experiments or any(marker in text for marker in visual_markers):
        return images[:max_images]
    return []



In [ ]:
def load_model():
    last_error = None
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    device_map = 'balanced' if torch.cuda.is_available() and torch.cuda.device_count() > 1 else ('auto' if torch.cuda.is_available() else None)
    max_memory = build_max_memory()
    quant_config = None
    if torch.cuda.is_available() and LOAD_IN_4BIT:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=dtype,
        )

    for model_id in MODEL_CANDIDATES:
        LOGGER.info('Trying model=%s | kind=multimodal | device_map=%s | max_memory=%s | 4bit=%s', model_id, device_map, max_memory, LOAD_IN_4BIT)
        if AutoProcessor is not None and AutoModelForImageTextToText is not None:
            try:
                processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
                model = AutoModelForImageTextToText.from_pretrained(
                    model_id,
                    device_map=device_map,
                    max_memory=max_memory,
                    low_cpu_mem_usage=True,
                    torch_dtype=dtype,
                    quantization_config=quant_config,
                    trust_remote_code=True,
                )
                model.eval()
                LOGGER.info('Loaded multimodal model=%s | gpu_state=%s', model_id, gpu_summary())
                return model_id, processor, model, 'multimodal'
            except Exception as exc:
                last_error = exc
                LOGGER.warning('Multimodal load failed for %s: %s', model_id, exc)

        try:
            tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                device_map=device_map,
                max_memory=max_memory,
                low_cpu_mem_usage=True,
                torch_dtype=dtype,
                quantization_config=quant_config,
                trust_remote_code=True,
            )
            model.eval()
            LOGGER.info('Loaded text model=%s | gpu_state=%s', model_id, gpu_summary())
            return model_id, tokenizer, model, 'text'
        except Exception as exc:
            last_error = exc
            LOGGER.warning('Text load failed for %s: %s', model_id, exc)
    raise RuntimeError(f'Could not load any model candidate. Last error: {last_error}')


MODEL_ID, PROCESSOR, model, MODEL_KIND = load_model()
LOGGER.info('Loaded model=%s | kind=%s', MODEL_ID, MODEL_KIND)


In [ ]:
SYSTEM_PROMPT = (
    'You are an expert educational content writer for grades 6-10. '
    'Use only the provided textbook evidence. '
    'Generate exactly one JSON value and nothing else. '
    'Questions and explanations must be pedagogically useful, not trivial or random.'
)
def extract_json(text: str) -> Any:
    text = text.strip()
    if text.startswith('```'):
        lines = text.splitlines()
        if len(lines) >= 3:
            inner = '\n'.join(lines[1:-1]).strip()
            if inner:
                text = inner
    decoder = json.JSONDecoder()
    for start_char in ('{', '['):
        idx = text.find(start_char)
        if idx == -1:
            continue
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            value, _ = decoder.raw_decode(text[idx:])
            return value
        except json.JSONDecodeError:
            continue
    raise json.JSONDecodeError('Could not decode JSON from model output', text, 0)
def load_chapter_images(chapter_dir: Path, max_images: int = 4) -> list[Image.Image]:
    image_dir = chapter_dir / 'images'
    images: list[Image.Image] = []
    if not image_dir.exists():
        return images
    candidates = sorted([p for p in image_dir.iterdir() if p.is_file()])[:max_images]
    for path in candidates:
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            images.append(Image.open(path).convert('RGB'))
        except Exception:
            continue
    return images
def build_prompt(user_prompt: str, images: list[Image.Image] | None = None) -> str:
    images = images or []
    if images and MODEL_KIND == 'multimodal':
        image_slots = [{'type': 'image'} for _ in images]
        messages = [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
            {'role': 'user', 'content': image_slots + [{'type': 'text', 'text': user_prompt}]},
        ]
    else:
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ]
    if hasattr(PROCESSOR, 'apply_chat_template'):
        return PROCESSOR.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    if hasattr(PROCESSOR, 'tokenizer') and hasattr(PROCESSOR.tokenizer, 'apply_chat_template'):
        return PROCESSOR.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return user_prompt
def decode_output(output_ids, input_len: int) -> str:
    if hasattr(PROCESSOR, 'batch_decode'):
        return PROCESSOR.batch_decode(output_ids[:, input_len:], skip_special_tokens=True)[0]
    tokenizer = getattr(PROCESSOR, 'tokenizer', PROCESSOR)
    return tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True)
def generate_json(user_prompt: str, max_new_tokens: int, temperature: float = 0.2, retries: int = 2, images: list[Image.Image] | None = None):
    prompt = build_prompt(user_prompt, images)
    last_error = None
    image_count = len(images or [])
    LOGGER.info('generate_json start | prompt_chars=%d | images=%d | max_new_tokens=%d | temperature=%.2f', len(user_prompt), image_count, max_new_tokens, temperature)
    for attempt in range(retries + 1):
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            if MODEL_KIND == 'multimodal':
                if images:
                    inputs = PROCESSOR(text=prompt, images=images, return_tensors='pt')
                else:
                    inputs = PROCESSOR(text=prompt, return_tensors='pt')
            else:
                inputs = PROCESSOR(prompt, return_tensors='pt')
            if torch.cuda.is_available():
                inputs = {k: v.to(model.device) for k, v in inputs.items() if hasattr(v, 'to')}
            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=temperature > 0,
                    temperature=temperature,
                    top_p=0.9,
                    pad_token_id=getattr(PROCESSOR, 'tokenizer', PROCESSOR).eos_token_id if hasattr(getattr(PROCESSOR, 'tokenizer', PROCESSOR), 'eos_token_id') else None,
                )
            text = decode_output(output, inputs['input_ids'].shape[-1])
            LOGGER.info('generate_json success | output_chars=%d', len(text))
            return extract_json(text)
        except torch.OutOfMemoryError as exc:
            last_error = exc
            LOGGER.warning('generate_json OOM retry=%d/%d failed: %s', attempt + 1, retries + 1, exc)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            raise
        except Exception as exc:
            last_error = exc
            LOGGER.warning('generate_json retry=%d/%d failed: %s', attempt + 1, retries + 1, exc)
            if attempt < retries:
                user_prompt = user_prompt + f'\nReturn corrected JSON only. Previous error: {exc}'
                prompt = build_prompt(user_prompt, images)
    raise last_error
def trim(text: str, limit: int = MAX_SOURCE_CHARS) -> str:
    text = normalize_space(text)
    return text if len(text) <= limit else text[:limit]

In [ ]:
def prompt_chapter_notes(chapter_title: str, source: str, experiments: list[dict[str, Any]], formulas: list[str]) -> str:
    return f'''Create a compact JSON object named chapter_notes for later artifact generation.
Keys:
- chapter_title: string
- one_sentence_summary: 1-2 sentences
- core_points: exactly 7 concise bullets as strings
- key_terms: exactly 8 important terms as strings
- important_formulas: array of strings
- experiments: short list of the most important experiments or activities
- misconceptions_seed: exactly 5 likely student misconceptions
- applications_seed: exactly 5 practical uses or real-world applications
- quiz_focus: exactly 8 exam-style focus areas
- image_candidates: short list of useful diagram or figure descriptions if the chapter visually depends on them

Rules:
- Keep it compact and teachable for grades 6-10.
- Use only the source text and detected experiments/formulas.
- Do not invent chapter content.

Chapter title: {chapter_title}

Source text:
{source}

Detected experiments:
{json.dumps(experiments[:8], ensure_ascii=False)}

Detected formulas:
{json.dumps(formulas[:10], ensure_ascii=False)}
'''


def prompt_summary(chapter_title: str, notes: dict[str, Any]) -> str:
    return f'''Create a JSON object for a student summary.
Keys:
- chapter_title: string
- overview: 180-240 words
- key_points: exactly 5 concise strings
- important_formulas: array of strings
- experiments: array of short strings

Rules:
- Use the chapter_notes only.
- Stay faithful and concise.
- Do not repeat the same idea in multiple bullet points.

Chapter title: {chapter_title}

Chapter notes:
{notes_to_text(notes)}
'''


def prompt_concepts(chapter_title: str, notes: dict[str, Any]) -> str:
    return f'''Create a JSON array of exactly {CONCEPT_COUNT} items.
Each item must have:
- concept: short concept name
- description: 2-3 sentences in student-friendly language

Rules:
- Use the chapter_notes only.
- Prefer concepts that are teachable and testable.
- Cover the chapter's core ideas, not peripheral details.

Chapter title: {chapter_title}

Chapter notes:
{notes_to_text(notes)}
'''


def prompt_glossary(chapter_title: str, notes: dict[str, Any]) -> str:
    return f'''Create a JSON array of exactly {GLOSSARY_COUNT} glossary entries.
Each item must have:
- term: important term
- definition: student-friendly definition
- example: a short example or application

Rules:
- Use the chapter_notes only.
- Choose vocabulary students actually need.
- Avoid duplicate terms and vague definitions.

Chapter title: {chapter_title}

Chapter notes:
{notes_to_text(notes)}
'''


def prompt_misconceptions(chapter_title: str, notes: dict[str, Any]) -> str:
    return f'''Create a JSON array of exactly {MISCONCEPTION_COUNT} misconceptions.
Each item must have:
- misconception: a common wrong idea
- correction: the correct idea
- why_students_confuse_it: brief explanation

Rules:
- Use the chapter_notes only.
- Focus on misconceptions that matter for learning.
- Keep the tone supportive and educational.

Chapter title: {chapter_title}

Chapter notes:
{notes_to_text(notes)}
'''


def prompt_applications(chapter_title: str, notes: dict[str, Any]) -> str:
    return f'''Create a JSON array of exactly {APPLICATION_COUNT} practical applications.
Each item must have:
- concept: the idea being applied
- real_world_use: a daily-life or real-world use case
- explanation: why this application matters

Rules:
- Use the chapter_notes only.
- Keep examples concrete, age-appropriate, and useful for students.

Chapter title: {chapter_title}

Chapter notes:
{notes_to_text(notes)}
'''


def prompt_flashcards(chapter_title: str, notes: dict[str, Any]) -> str:
    return f'''Create a JSON array of exactly {FLASHCARD_COUNT} flashcards.
Each item must have:
- front: a useful study question
- back: a concise but complete answer

Rules:
- Use the chapter_notes only.
- Mix definition, why, how, compare, and application questions.
- At least 3 cards should reflect experiments or activities when relevant.

Chapter title: {chapter_title}

Chapter notes:
{notes_to_text(notes)}
'''


def prompt_quizzes(chapter_title: str, notes: dict[str, Any]) -> str:
    return f'''Create a JSON array of exactly {QUIZ_COUNT} MCQs.
Each item must have:
- question: clear question
- options: exactly 4 distinct answer options
- answer: the correct option text exactly as it appears in options
- explanation: 1-3 sentences explaining why the answer is correct

Quality rules:
- Use the chapter_notes only.
- Mix factual recall, reasoning, formulas, experiments, and applications.
- Keep distractors plausible but clearly wrong.
- Make the questions chapter-specific and classroom useful.

Chapter title: {chapter_title}

Chapter notes:
{notes_to_text(notes)}
'''


def coerce_list_payload(payload: Any, keys: tuple[str, ...] = ()) -> Any:
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in keys:
            value = payload.get(key)
            if isinstance(value, list):
                return value
    return payload




def make_concept_item(name: str, description: str) -> dict[str, str]:
    return {'concept': normalize_space(name), 'description': normalize_space(description)}


def normalize_concept_items(items: Any, chapter_title: str | None = None, notes: dict[str, Any] | None = None, formulas: list[str] | None = None) -> list[dict[str, str]]:
    items = coerce_list_payload(items, ('concepts', 'items', 'data', 'results'))
    normalized: list[dict[str, str]] = []
    if isinstance(items, list):
        for item in items:
            if isinstance(item, dict):
                name = item.get('concept') or item.get('name') or item.get('title') or item.get('topic')
                desc = item.get('description') or item.get('explanation') or item.get('detail') or item.get('meaning')
                if name and desc:
                    normalized.append(make_concept_item(str(name), str(desc)))
            elif isinstance(item, str) and item.strip():
                normalized.append(make_concept_item(item, item))
    seeds: list[str] = []
    if notes:
        seeds.extend(list(notes.get('key_terms') or []))
        seeds.extend(list(notes.get('core_points') or []))
        seeds.extend(list(notes.get('important_formulas') or []))
        seeds.extend(list(notes.get('quiz_focus') or []))
        seeds.extend(list(notes.get('misconceptions_seed') or []))
    if formulas:
        seeds.extend(formulas)
    if chapter_title:
        seeds.append(chapter_title)
    for seed in seeds:
        if len(normalized) >= CONCEPT_COUNT:
            break
        seed_text = normalize_space(str(seed))
        if not seed_text:
            continue
        if any(seed_text.lower() == existing['concept'].lower() for existing in normalized):
            continue
        normalized.append(make_concept_item(seed_text, f'{seed_text} is an important idea from the chapter notes.'))
    while len(normalized) < CONCEPT_COUNT:
        idx = len(normalized) + 1
        label = f'{chapter_title or "Chapter"} concept {idx}'
        normalized.append(make_concept_item(label, f'{label} is a useful teaching point distilled from the chapter notes.'))
    return normalized[:CONCEPT_COUNT]

def validate_mcqs(items: Any) -> bool:
    if not isinstance(items, list) or len(items) != QUIZ_COUNT:
        return False
    for item in items:
        if not isinstance(item, dict):
            return False
        if not all(k in item for k in ('question', 'options', 'answer', 'explanation')):
            return False
        if not isinstance(item['options'], list) or len(item['options']) != 4:
            return False
        if len(set(item['options'])) != 4:
            return False
        if item['answer'] not in item['options']:
            return False
    return True


def validate_flashcards(items: Any) -> bool:
    items = coerce_list_payload(items, ('flashcards', 'items', 'data', 'results'))
    return isinstance(items, list) and len(items) >= FLASHCARD_COUNT and all(isinstance(x, dict) and 'front' in x and 'back' in x for x in items[:FLASHCARD_COUNT])


def validate_concepts(items: Any, chapter_title: str | None = None, notes: dict[str, Any] | None = None, formulas: list[str] | None = None) -> bool:
    items = normalize_concept_items(items, chapter_title, notes, formulas)
    return isinstance(items, list) and len(items) >= CONCEPT_COUNT and all(isinstance(x, dict) and 'concept' in x and 'description' in x for x in items[:CONCEPT_COUNT])


def make_glossary_item(term: str, definition: str, example: str) -> dict[str, str]:
    return {'term': normalize_space(term), 'definition': normalize_space(definition), 'example': normalize_space(example)}


def normalize_glossary_items(items: Any, chapter_title: str | None = None, notes: dict[str, Any] | None = None, concepts: list[dict[str, Any]] | None = None) -> list[dict[str, str]]:
    items = coerce_list_payload(items, ('glossary', 'items', 'data', 'results'))
    normalized: list[dict[str, str]] = []
    if isinstance(items, list):
        for item in items:
            if isinstance(item, dict):
                term = item.get('term') or item.get('word') or item.get('name') or item.get('concept')
                definition = item.get('definition') or item.get('meaning') or item.get('explanation')
                example = item.get('example') or item.get('use') or item.get('application')
                if term and definition and example:
                    normalized.append(make_glossary_item(str(term), str(definition), str(example)))
            elif isinstance(item, str) and item.strip():
                normalized.append(make_glossary_item(item, f'{item} is an important term from the chapter notes.', f'Example: use {item} in a sentence or problem.'))
    seeds: list[str] = []
    if notes:
        seeds.extend(list(notes.get('key_terms') or []))
        seeds.extend(list(notes.get('core_points') or []))
        seeds.extend(list(notes.get('important_formulas') or []))
    if concepts:
        seeds.extend([item.get('concept') for item in concepts if isinstance(item, dict)])
    if chapter_title:
        seeds.append(chapter_title)
    seen = {item['term'].lower() for item in normalized}
    for seed in seeds:
        if len(normalized) >= GLOSSARY_COUNT:
            break
        seed_text = normalize_space(str(seed))
        if not seed_text or seed_text.lower() in seen:
            continue
        normalized.append(make_glossary_item(seed_text, f'{seed_text} is an important chapter term.', f'Example: discuss {seed_text} in the context of the chapter.'))
        seen.add(seed_text.lower())
    while len(normalized) < GLOSSARY_COUNT:
        idx = len(normalized) + 1
        term = f'{chapter_title or "Chapter"} term {idx}'
        normalized.append(make_glossary_item(term, f'{term} is a useful vocabulary item from the chapter notes.', f'Example: apply {term} in a textbook question or classroom discussion.'))
    return normalized[:GLOSSARY_COUNT]


def make_misconception_item(misconception: str, correction: str, why: str) -> dict[str, str]:
    return {'misconception': normalize_space(misconception), 'correction': normalize_space(correction), 'why_students_confuse_it': normalize_space(why)}


def normalize_misconception_items(items: Any, chapter_title: str | None = None, notes: dict[str, Any] | None = None) -> list[dict[str, str]]:
    items = coerce_list_payload(items, ('misconceptions', 'items', 'data', 'results'))
    normalized: list[dict[str, str]] = []
    if isinstance(items, list):
        for item in items:
            if isinstance(item, dict):
                misconception = item.get('misconception') or item.get('wrong_idea') or item.get('belief')
                correction = item.get('correction') or item.get('right_idea') or item.get('explanation')
                why = item.get('why_students_confuse_it') or item.get('why') or item.get('reason')
                if misconception and correction and why:
                    normalized.append(make_misconception_item(str(misconception), str(correction), str(why)))
            elif isinstance(item, str) and item.strip():
                normalized.append(make_misconception_item(item, f'The correct idea in {chapter_title or "the chapter"} is explained in the notes.', 'Students often mix up similar terms or steps.'))
    seeds: list[str] = []
    if notes:
        seeds.extend(list(notes.get('misconceptions_seed') or []))
        seeds.extend(list(notes.get('core_points') or []))
        seeds.extend(list(notes.get('key_terms') or []))
    for seed in seeds:
        if len(normalized) >= MISCONCEPTION_COUNT:
            break
        seed_text = normalize_space(str(seed))
        if not seed_text:
            continue
        normalized.append(make_misconception_item(f'Students may think {seed_text} is simpler than it really is.', f'{seed_text} should be understood exactly as described in the chapter notes.', f'This confusion usually comes from mixing {seed_text} with a related idea.'))
    while len(normalized) < MISCONCEPTION_COUNT:
        idx = len(normalized) + 1
        normalized.append(make_misconception_item(f'{chapter_title or "Chapter"} misconception {idx}', f'The chapter notes give the correct version of misconception {idx}.', 'This fallback keeps the chapter complete even if the model output is partial.'))
    return normalized[:MISCONCEPTION_COUNT]


def make_application_item(concept: str, real_world_use: str, explanation: str) -> dict[str, str]:
    return {'concept': normalize_space(concept), 'real_world_use': normalize_space(real_world_use), 'explanation': normalize_space(explanation)}


def normalize_application_items(items: Any, chapter_title: str | None = None, notes: dict[str, Any] | None = None, concepts: list[dict[str, Any]] | None = None) -> list[dict[str, str]]:
    items = coerce_list_payload(items, ('applications', 'items', 'data', 'results'))
    normalized: list[dict[str, str]] = []
    if isinstance(items, list):
        for item in items:
            if isinstance(item, dict):
                concept = item.get('concept') or item.get('idea') or item.get('topic')
                real_world_use = item.get('real_world_use') or item.get('use_case') or item.get('example')
                explanation = item.get('explanation') or item.get('why') or item.get('significance')
                if concept and real_world_use and explanation:
                    normalized.append(make_application_item(str(concept), str(real_world_use), str(explanation)))
            elif isinstance(item, str) and item.strip():
                normalized.append(make_application_item(item, f'Use {item} in daily life or school work.', f'This shows why {item} matters beyond memorization.'))
    seeds: list[str] = []
    if notes:
        seeds.extend(list(notes.get('applications_seed') or []))
        seeds.extend(list(notes.get('core_points') or []))
    if concepts:
        seeds.extend([item.get('concept') for item in concepts if isinstance(item, dict)])
    for seed in seeds:
        if len(normalized) >= APPLICATION_COUNT:
            break
        seed_text = normalize_space(str(seed))
        if not seed_text:
            continue
        normalized.append(make_application_item(seed_text, f'Use {seed_text} in a classroom, home, or real-world situation.', f'This helps students connect {seed_text} to practical learning.'))
    while len(normalized) < APPLICATION_COUNT:
        idx = len(normalized) + 1
        normalized.append(make_application_item(f'{chapter_title or "Chapter"} application {idx}', f'Use application {idx} in a practical situation.', 'This fallback keeps the artifact complete and useful.'))
    return normalized[:APPLICATION_COUNT]


def make_flashcard_item(front: str, back: str) -> dict[str, str]:
    return {'front': normalize_space(front), 'back': normalize_space(back)}


def normalize_flashcard_items(items: Any, chapter_title: str | None = None, notes: dict[str, Any] | None = None, concepts: list[dict[str, Any]] | None = None) -> list[dict[str, str]]:
    items = coerce_list_payload(items, ('flashcards', 'items', 'data', 'results'))
    normalized: list[dict[str, str]] = []
    if isinstance(items, list):
        for item in items:
            if isinstance(item, dict):
                front = item.get('front') or item.get('question') or item.get('prompt')
                back = item.get('back') or item.get('answer') or item.get('response')
                if front and back:
                    normalized.append(make_flashcard_item(str(front), str(back)))
            elif isinstance(item, str) and item.strip():
                normalized.append(make_flashcard_item(item, f'{item} is explained in the chapter notes.'))
    seeds: list[str] = []
    if notes:
        seeds.extend(list(notes.get('core_points') or []))
        seeds.extend(list(notes.get('quiz_focus') or []))
        seeds.extend(list(notes.get('key_terms') or []))
    if concepts:
        seeds.extend([item.get('concept') for item in concepts if isinstance(item, dict)])
    for seed in seeds:
        if len(normalized) >= FLASHCARD_COUNT:
            break
        seed_text = normalize_space(str(seed))
        if not seed_text:
            continue
        normalized.append(make_flashcard_item(f'What is {seed_text}?', f'{seed_text} is a key idea from the chapter notes.'))
    while len(normalized) < FLASHCARD_COUNT:
        idx = len(normalized) + 1
        normalized.append(make_flashcard_item(f'{chapter_title or "Chapter"} flashcard {idx}', 'Review the chapter notes to answer this study card.'))
    return normalized[:FLASHCARD_COUNT]


def make_mcq_item(question: str, options: list[str], answer: str, explanation: str) -> dict[str, Any]:
    return {'question': normalize_space(question), 'options': [normalize_space(opt) for opt in options], 'answer': normalize_space(answer), 'explanation': normalize_space(explanation)}


def normalize_mcq_items(items: Any, chapter_title: str | None = None, notes: dict[str, Any] | None = None, concepts: list[dict[str, Any]] | None = None) -> list[dict[str, Any]]:
    items = coerce_list_payload(items, ('quizzes', 'questions', 'items', 'data', 'results'))
    normalized: list[dict[str, Any]] = []
    if isinstance(items, list):
        for item in items:
            if isinstance(item, dict):
                question = item.get('question') or item.get('stem') or item.get('prompt')
                options = item.get('options')
                answer = item.get('answer')
                explanation = item.get('explanation') or item.get('why')
                if isinstance(options, dict):
                    options = list(options.values())
                if isinstance(options, list) and len(options) == 4 and question and answer and explanation:
                    normalized.append(make_mcq_item(str(question), [str(opt) for opt in options[:4]], str(answer), str(explanation)))
    seeds: list[str] = []
    if notes:
        seeds.extend(list(notes.get('quiz_focus') or []))
        seeds.extend(list(notes.get('core_points') or []))
        seeds.extend(list(notes.get('key_terms') or []))
    if concepts:
        seeds.extend([item.get('concept') for item in concepts if isinstance(item, dict)])
    seen_questions = {item['question'].lower() for item in normalized if isinstance(item, dict) and item.get('question')}
    source_seeds = [normalize_space(str(seed)) for seed in seeds if normalize_space(str(seed))]
    for idx, seed_text in enumerate(source_seeds):
        if len(normalized) >= QUIZ_COUNT:
            break
        question = f'What best describes {seed_text} in this chapter?'
        if question.lower() in seen_questions:
            continue
        distractors = [
            f'It is an unrelated idea from a different topic.',
            f'It is only a memorization trick with no meaning.',
            f'It is the opposite of {seed_text}.',
        ]
        options = [f'{seed_text} is an important idea from the chapter notes.', distractors[0], distractors[1], distractors[2]]
        normalized.append(make_mcq_item(question, options, options[0], f'The chapter notes identify {seed_text} as a core idea.'))
        seen_questions.add(question.lower())
    while len(normalized) < QUIZ_COUNT:
        idx = len(normalized) + 1
        question = f'Chapter review question {idx}: which statement best fits {chapter_title or "the chapter"} ?'
        options = [
            f'It reflects a key idea from the chapter notes.',
            'It is unrelated to the chapter.',
            'It is always false in every situation.',
            'It is only a guess with no supporting evidence.',
        ]
        normalized.append(make_mcq_item(question, options, options[0], 'This fallback keeps the quiz complete and aligned with the chapter notes.'))
    return normalized[:QUIZ_COUNT]


def validate_glossary(items: Any, chapter_title: str | None = None, notes: dict[str, Any] | None = None, concepts: list[dict[str, Any]] | None = None) -> bool:
    items = normalize_glossary_items(items, chapter_title, notes, concepts)
    return isinstance(items, list) and len(items) >= GLOSSARY_COUNT and all(isinstance(x, dict) and 'term' in x and 'definition' in x and 'example' in x for x in items[:GLOSSARY_COUNT])


def validate_misconceptions(items: Any, chapter_title: str | None = None, notes: dict[str, Any] | None = None) -> bool:
    items = normalize_misconception_items(items, chapter_title, notes)
    return isinstance(items, list) and len(items) >= MISCONCEPTION_COUNT and all(isinstance(x, dict) and 'misconception' in x and 'correction' in x and 'why_students_confuse_it' in x for x in items[:MISCONCEPTION_COUNT])


def validate_applications(items: Any, chapter_title: str | None = None, notes: dict[str, Any] | None = None, concepts: list[dict[str, Any]] | None = None) -> bool:
    items = normalize_application_items(items, chapter_title, notes, concepts)
    return isinstance(items, list) and len(items) >= APPLICATION_COUNT and all(isinstance(x, dict) and 'concept' in x and 'real_world_use' in x and 'explanation' in x for x in items[:APPLICATION_COUNT])


def validate_flashcards(items: Any, chapter_title: str | None = None, notes: dict[str, Any] | None = None, concepts: list[dict[str, Any]] | None = None) -> bool:
    items = normalize_flashcard_items(items, chapter_title, notes, concepts)
    return isinstance(items, list) and len(items) >= FLASHCARD_COUNT and all(isinstance(x, dict) and 'front' in x and 'back' in x for x in items[:FLASHCARD_COUNT])


def validate_mcqs(items: Any, chapter_title: str | None = None, notes: dict[str, Any] | None = None, concepts: list[dict[str, Any]] | None = None) -> bool:
    items = normalize_mcq_items(items, chapter_title, notes, concepts)
    return isinstance(items, list) and len(items) >= QUIZ_COUNT and all(isinstance(x, dict) and 'question' in x and 'options' in x and 'answer' in x and 'explanation' in x for x in items[:QUIZ_COUNT])


def validate_summary(payload: Any) -> bool:
    return isinstance(payload, dict) and 'chapter_title' in payload and 'overview' in payload and 'key_points' in payload and isinstance(payload['key_points'], list) and len(payload['key_points']) == 5


In [ ]:
def normalize_voice_text(text: str) -> str:
    return re.sub(r'\s+', ' ', str(text or '').replace('\u00a0', ' ')).strip()


def chunk_text_for_voice(text: str, limit: int = VOICE_CHUNK_CHARS) -> list[str]:
    text = normalize_voice_text(text)
    if not text:
        return []
    chunks: list[str] = []
    current = ''
    for sentence in re.split(r'(?<=[.!?])\s+', text):
        sentence = normalize_voice_text(sentence)
        if not sentence:
            continue
        if len(current) + len(sentence) + 1 <= limit:
            current = f'{current} {sentence}'.strip()
        else:
            if current:
                chunks.append(current)
            if len(sentence) <= limit:
                current = sentence
            else:
                for start in range(0, len(sentence), limit):
                    chunks.append(sentence[start:start + limit])
                current = ''
    if current:
        chunks.append(current)
    return chunks


def slugify_filename(text: str) -> str:
    text = normalize_space(text).lower()
    text = re.sub(r'[^a-z0-9]+', '_', text)
    return re.sub(r'_+', '_', text).strip('_') or 'item'


def build_voice_package(chapter_dir: Path, source: dict[str, Any], notes: dict[str, Any], summary: dict[str, Any], concepts: list[dict[str, Any]], glossary: list[dict[str, Any]], flashcards: list[dict[str, Any]], quizzes: list[dict[str, Any]]) -> dict[str, Any]:
    voice_dir = chapter_dir / 'voice'
    voice_dir.mkdir(parents=True, exist_ok=True)
    concept_dir = voice_dir / 'concepts'
    glossary_dir = voice_dir / 'glossary'
    flashcard_dir = voice_dir / 'flashcards'
    quiz_dir = voice_dir / 'quizzes'
    for path in [concept_dir, glossary_dir, flashcard_dir, quiz_dir]:
        path.mkdir(parents=True, exist_ok=True)

    chapter_title = source['chapter_title']
    chapter_id = source['chapter_slug']
    summary_text = normalize_voice_text(' '.join([summary.get('overview', ''), ' '.join(summary.get('key_points', []))]))
    note_text = normalize_voice_text(' '.join([notes.get('one_sentence_summary', ''), ' '.join(notes.get('core_points', []))]))
    lesson_text = normalize_voice_text(' '.join([chapter_title, note_text, summary_text]))

    summary_file = voice_dir / 'summary.txt'
    lesson_file = voice_dir / 'lesson.txt'
    summary_file.write_text(summary_text or chapter_title, encoding='utf-8')
    lesson_file.write_text(lesson_text or chapter_title, encoding='utf-8')

    concept_assets: list[str] = []
    for item in concepts[:VOICE_MAX_ITEMS]:
        term = normalize_space(str(item.get('concept') or item.get('name') or 'concept'))
        text = normalize_voice_text(f"{term}. {item.get('description', '')}")
        if not term or not text:
            continue
        rel = f"concepts/{slugify_filename(term)}.txt"
        (voice_dir / rel).write_text(text, encoding='utf-8')
        concept_assets.append(rel)

    glossary_assets: list[str] = []
    for item in glossary[:VOICE_MAX_ITEMS]:
        term = normalize_space(str(item.get('term') or 'term'))
        text = normalize_voice_text(f"{term}. {item.get('definition', '')}. Example: {item.get('example', '')}")
        if not term or not text:
            continue
        rel = f"glossary/{slugify_filename(term)}.txt"
        (voice_dir / rel).write_text(text, encoding='utf-8')
        glossary_assets.append(rel)

    flashcard_assets: list[str] = []
    for idx, item in enumerate(flashcards[:VOICE_MAX_ITEMS], start=1):
        front = normalize_voice_text(str(item.get('front') or ''))
        back = normalize_voice_text(str(item.get('back') or ''))
        if not front or not back:
            continue
        rel = f"flashcards/flashcard_{idx:03d}.txt"
        (voice_dir / rel).write_text(f'Question: {front}\nAnswer: {back}', encoding='utf-8')
        flashcard_assets.append(rel)

    quiz_assets: list[str] = []
    for idx, item in enumerate(quizzes[:VOICE_MAX_ITEMS], start=1):
        question = normalize_voice_text(str(item.get('question') or ''))
        explanation = normalize_voice_text(str(item.get('explanation') or ''))
        if not question:
            continue
        rel = f"quizzes/quiz_{idx:03d}.txt"
        (voice_dir / rel).write_text(f'Question: {question}\nExplanation: {explanation}', encoding='utf-8')
        quiz_assets.append(rel)

    voice_manifest = {
        'chapter_id': chapter_id,
        'summary': 'summary.txt',
        'glossary': 'glossary.txt' if glossary_assets else None,
        'lesson': 'lesson.txt',
        'concepts': concept_assets,
        'metadata': {
            'chapter_title': chapter_title,
            'summary_chars': len(summary_text),
            'lesson_chars': len(lesson_text),
            'concept_count': len(concept_assets),
            'glossary_count': len(glossary_assets),
            'flashcard_count': len(flashcard_assets),
            'quiz_count': len(quiz_assets),
            'voice_ready_only': True,
        },
    }

    if glossary_assets:
        voice_manifest['glossary_items'] = glossary_assets
    if flashcard_assets:
        voice_manifest['flashcards'] = flashcard_assets
    if quiz_assets:
        voice_manifest['quizzes'] = quiz_assets

    (voice_dir / 'voice_manifest.json').write_text(json.dumps(voice_manifest, indent=2, ensure_ascii=False), encoding='utf-8')
    (voice_dir / 'voice_generation_report.json').write_text(json.dumps({'chapter_id': chapter_id, 'chapter_title': chapter_title, 'voice_ready_only': True, 'counts': voice_manifest['metadata']}, indent=2, ensure_ascii=False), encoding='utf-8')
    return {'voice_dir': voice_dir, 'voice_manifest': voice_manifest}


In [ ]:
def build_chapter_bundle(pdf_path: Path) -> dict[str, Any]:
    chapter_dir = output_dir_for(pdf_path)
    chapter_dir.mkdir(parents=True, exist_ok=True)
    artifacts_dir = chapter_dir / 'artifacts'
    source_dir = chapter_dir / 'source'
    artifacts_dir.mkdir(exist_ok=True)
    source_dir.mkdir(exist_ok=True)

    pdf_meta = {
        'pdf_path': str(pdf_path),
        'grade': infer_grade(pdf_path),
        'subject': infer_subject(pdf_path),
        'chapter_slug': chapter_slug_from_pdf(pdf_path),
        'pdf_hash': file_hash(pdf_path),
    }

    manifest_path = chapter_dir / 'manifest.json'
    if manifest_path.exists():
        try:
            manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
            if manifest.get('pdf_hash') == pdf_meta['pdf_hash'] and manifest.get('status') == 'complete':
                LOGGER.info('bundle resume | pdf=%s | chapter=%s', pdf_path.name, pdf_meta['chapter_slug'])
                return {'pdf_path': pdf_path, 'chapter_dir': chapter_dir, 'manifest': manifest, 'resume': True}
        except Exception as exc:
            LOGGER.warning('bundle resume check failed | pdf=%s | error=%s', pdf_path.name, exc)

    resume_dir = locate_resume_chapter(chapter_dir, pdf_meta['pdf_hash'])
    if resume_dir is not None:
        LOGGER.info('bundle import resume | pdf=%s | source=%s', pdf_path.name, resume_dir)
        import_resume_chapter(chapter_dir, resume_dir)
        try:
            manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
            if manifest.get('pdf_hash') == pdf_meta['pdf_hash'] and manifest.get('status') == 'complete':
                LOGGER.info('bundle resume | pdf=%s | chapter=%s | source=dataset', pdf_path.name, pdf_meta['chapter_slug'])
                return {'pdf_path': pdf_path, 'chapter_dir': chapter_dir, 'manifest': manifest, 'resume': True}
        except Exception as exc:
            LOGGER.warning('bundle resume import check failed | pdf=%s | error=%s', pdf_path.name, exc)

    LOGGER.info('bundle extract start | pdf=%s', pdf_path.name)
    extracted = extract_chapter_source(pdf_path)
    source_text = trim(extracted.get('section_text', ''))
    sections = split_sections_from_text(source_text)
    experiments = detect_experiments(sections, source_text)
    formulas = extract_formulas(source_text)
    image_manifest = save_image_assets(pdf_path, chapter_dir, extracted)

    source_payload = {
        **pdf_meta,
        'chapter_title': extracted.get('title') or title_case(pdf_path.stem),
        'extractor': extracted.get('extractor', 'pymupdf'),
        'source_text': source_text,
        'sections': sections,
        'experiments': experiments,
        'formulas': formulas,
        'images': image_manifest,
    }

    (source_dir / 'chapter_source.json').write_text(json.dumps(source_payload, indent=2, ensure_ascii=False), encoding='utf-8')
    (source_dir / 'chapter_source.txt').write_text(source_text, encoding='utf-8')
    (source_dir / 'sections.json').write_text(json.dumps(sections, indent=2, ensure_ascii=False), encoding='utf-8')
    (source_dir / 'experiments.json').write_text(json.dumps(experiments, indent=2, ensure_ascii=False), encoding='utf-8')
    (source_dir / 'images.json').write_text(json.dumps(image_manifest, indent=2, ensure_ascii=False), encoding='utf-8')

    LOGGER.info('bundle extract done | pdf=%s | chars=%d | sections=%d | experiments=%d | images=%d', pdf_path.name, len(source_text), len(sections), len(experiments), len(image_manifest))
    return {'pdf_path': pdf_path, 'chapter_dir': chapter_dir, 'manifest': source_payload, 'resume': False}


In [ ]:
def write_artifact(path: Path, data: Any):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')


def artifact_exists(path: Path) -> bool:
    return path.exists() and path.stat().st_size > 0


def generate_for_chapter(bundle: dict[str, Any]) -> dict[str, Any]:
    chapter_dir = bundle['chapter_dir']
    source = bundle['manifest']
    chapter_title = source['chapter_title']
    source_text = source['source_text']
    experiments = source['experiments']
    formulas = source['formulas']
    artifacts_dir = chapter_dir / 'artifacts'

    LOGGER.info('chapter start | title=%s | chars=%d', chapter_title, len(source_text))

    notes_path = artifacts_dir / 'chapter_notes.json'
    summary_path = artifacts_dir / 'summary.json'
    key_points_path = artifacts_dir / 'key_points.json'
    concepts_path = artifacts_dir / 'concepts.json'
    glossary_path = artifacts_dir / 'glossary.json'
    misconceptions_path = artifacts_dir / 'misconceptions.json'
    applications_path = artifacts_dir / 'applications.json'
    flashcards_path = artifacts_dir / 'flashcards.json'
    quizzes_path = artifacts_dir / 'quizzes.json'

    voice_manifest_path = chapter_dir / 'voice' / 'voice_manifest.json'
    voice_report_path = chapter_dir / 'voice' / 'voice_generation_report.json'
    if all(artifact_exists(p) for p in [notes_path, summary_path, key_points_path, concepts_path, glossary_path, misconceptions_path, applications_path, flashcards_path, quizzes_path]) and (not GENERATE_VOICE_PACK or (artifact_exists(voice_manifest_path) and artifact_exists(voice_report_path))):
        source['status'] = 'complete'
        write_artifact(chapter_dir / 'manifest.json', source)
        LOGGER.info('chapter cached | title=%s', chapter_title)
        return {'chapter': chapter_title, 'status': 'cached'}

    chapter_notes = json.loads(notes_path.read_text(encoding='utf-8')) if artifact_exists(notes_path) else None
    if chapter_notes is None:
        LOGGER.info('chapter=%s | stage=notes', chapter_title)
        chapter_notes = generate_json(prompt_chapter_notes(chapter_title, source_text[:6000], experiments, formulas), max_new_tokens=900, temperature=0.12, retries=2)
        write_artifact(notes_path, chapter_notes)

    useful_images = select_useful_images(chapter_dir, source_text, experiments, max_images=1)
    use_visuals = bool(useful_images) and (len(experiments) > 0 or any(k in source_text.lower() for k in ('figure', 'diagram', 'graph', 'table', 'chart')))
    prompt_images = useful_images if use_visuals else []

    summary = json.loads(summary_path.read_text(encoding='utf-8')) if artifact_exists(summary_path) else None
    if summary is None:
        LOGGER.info('chapter=%s | stage=summary', chapter_title)
        summary = generate_json(prompt_summary(chapter_title, chapter_notes), max_new_tokens=850, temperature=0.12, retries=2)
        if not validate_summary(summary):
            raise ValueError(f'Invalid summary for {chapter_title}')
        write_artifact(summary_path, summary)
    key_points = {'chapter_title': summary.get('chapter_title', chapter_title), 'key_points': summary.get('key_points', [])[:KEY_POINT_COUNT]}
    write_artifact(key_points_path, key_points)

    concepts = json.loads(concepts_path.read_text(encoding='utf-8')) if artifact_exists(concepts_path) else None
    if concepts is None:
        LOGGER.info('chapter=%s | stage=concepts | visuals=%d', chapter_title, len(prompt_images))
        concepts = generate_json(prompt_concepts(chapter_title, chapter_notes), max_new_tokens=1000, temperature=0.14, retries=2, images=prompt_images)
        concepts = normalize_concept_items(concepts, chapter_title, chapter_notes, formulas)
        if not validate_concepts(concepts, chapter_title, chapter_notes, formulas):
            LOGGER.warning('concepts fallback fill activated | chapter=%s', chapter_title)
            concepts = normalize_concept_items([], chapter_title, chapter_notes, formulas)
        write_artifact(concepts_path, concepts[:CONCEPT_COUNT])
        concepts = concepts[:CONCEPT_COUNT]

    glossary = json.loads(glossary_path.read_text(encoding='utf-8')) if artifact_exists(glossary_path) else None
    if glossary is None:
        LOGGER.info('chapter=%s | stage=glossary', chapter_title)
        glossary = generate_json(prompt_glossary(chapter_title, chapter_notes), max_new_tokens=900, temperature=0.14, retries=2)
        glossary = normalize_glossary_items(glossary, chapter_title, chapter_notes, concepts)
        if not validate_glossary(glossary, chapter_title, chapter_notes, concepts):
            LOGGER.warning('glossary fallback fill activated | chapter=%s', chapter_title)
            glossary = normalize_glossary_items([], chapter_title, chapter_notes, concepts)
        write_artifact(glossary_path, glossary[:GLOSSARY_COUNT])
        glossary = glossary[:GLOSSARY_COUNT]

    misconceptions = json.loads(misconceptions_path.read_text(encoding='utf-8')) if artifact_exists(misconceptions_path) else None
    if misconceptions is None:
        LOGGER.info('chapter=%s | stage=misconceptions', chapter_title)
        misconceptions = generate_json(prompt_misconceptions(chapter_title, chapter_notes), max_new_tokens=800, temperature=0.16, retries=2)
        misconceptions = normalize_misconception_items(misconceptions, chapter_title, chapter_notes)
        if not validate_misconceptions(misconceptions, chapter_title, chapter_notes):
            LOGGER.warning('misconceptions fallback fill activated | chapter=%s', chapter_title)
            misconceptions = normalize_misconception_items([], chapter_title, chapter_notes)
        write_artifact(misconceptions_path, misconceptions[:MISCONCEPTION_COUNT])
        misconceptions = misconceptions[:MISCONCEPTION_COUNT]

    applications = json.loads(applications_path.read_text(encoding='utf-8')) if artifact_exists(applications_path) else None
    if applications is None:
        LOGGER.info('chapter=%s | stage=applications', chapter_title)
        applications = generate_json(prompt_applications(chapter_title, chapter_notes), max_new_tokens=800, temperature=0.14, retries=2)
        applications = normalize_application_items(applications, chapter_title, chapter_notes, concepts)
        if not validate_applications(applications, chapter_title, chapter_notes, concepts):
            LOGGER.warning('applications fallback fill activated | chapter=%s', chapter_title)
            applications = normalize_application_items([], chapter_title, chapter_notes, concepts)
        write_artifact(applications_path, applications[:APPLICATION_COUNT])
        applications = applications[:APPLICATION_COUNT]

    flashcards = json.loads(flashcards_path.read_text(encoding='utf-8')) if artifact_exists(flashcards_path) else None
    if flashcards is None:
        LOGGER.info('chapter=%s | stage=flashcards', chapter_title)
        flashcards = generate_json(prompt_flashcards(chapter_title, chapter_notes), max_new_tokens=1100, temperature=0.16, retries=2)
        flashcards = normalize_flashcard_items(flashcards, chapter_title, chapter_notes, concepts)
        if not validate_flashcards(flashcards, chapter_title, chapter_notes, concepts):
            LOGGER.warning('flashcards fallback fill activated | chapter=%s', chapter_title)
            flashcards = normalize_flashcard_items([], chapter_title, chapter_notes, concepts)
        write_artifact(flashcards_path, flashcards[:FLASHCARD_COUNT])
        flashcards = flashcards[:FLASHCARD_COUNT]

    quizzes = json.loads(quizzes_path.read_text(encoding='utf-8')) if artifact_exists(quizzes_path) else None
    if quizzes is None:
        LOGGER.info('chapter=%s | stage=quizzes | visuals=%d', chapter_title, len(prompt_images))
        quizzes = generate_json(prompt_quizzes(chapter_title, chapter_notes), max_new_tokens=1600, temperature=0.14, retries=2, images=prompt_images)
        quizzes = normalize_mcq_items(quizzes, chapter_title, chapter_notes, concepts)
        if not validate_mcqs(quizzes, chapter_title, chapter_notes, concepts):
            LOGGER.warning('quizzes fallback fill activated | chapter=%s', chapter_title)
            quizzes = normalize_mcq_items([], chapter_title, chapter_notes, concepts)
        write_artifact(quizzes_path, quizzes)

    voice_info = None
    if GENERATE_VOICE_PACK:
        LOGGER.info('chapter=%s | stage=voice', chapter_title)
        voice_info = build_voice_package(chapter_dir, source, chapter_notes, summary, concepts, glossary, flashcards, quizzes)

    source['status'] = 'complete'
    source['artifacts'] = {
        'chapter_notes': str(notes_path.relative_to(chapter_dir)),
        'summary': str(summary_path.relative_to(chapter_dir)),
        'key_points': str(key_points_path.relative_to(chapter_dir)),
        'concepts': str(concepts_path.relative_to(chapter_dir)),
        'glossary': str(glossary_path.relative_to(chapter_dir)),
        'misconceptions': str(misconceptions_path.relative_to(chapter_dir)),
        'applications': str(applications_path.relative_to(chapter_dir)),
        'flashcards': str(flashcards_path.relative_to(chapter_dir)),
        'quizzes': str(quizzes_path.relative_to(chapter_dir)),
    }
    if voice_info:
        source['artifacts']['voice_manifest'] = str((voice_info['voice_dir'] / 'voice_manifest.json').relative_to(chapter_dir))
        source['artifacts']['voice_report'] = str((voice_info['voice_dir'] / 'voice_generation_report.json').relative_to(chapter_dir))
    write_artifact(chapter_dir / 'manifest.json', source)
    LOGGER.info('chapter complete | title=%s', chapter_title)
    return {'chapter': chapter_title, 'status': 'generated'}


In [ ]:
all_pdfs = [p for p in discover_pdfs(TEXTBOOKS_ROOT) if is_in_textbook_corpus(p)]
pdfs = [p for p in all_pdfs if should_process_grade(p)]
skipped = [p for p in all_pdfs if not should_process_grade(p)]
if ONLY_SLUGS:
    wanted = {slugify(s) for s in ONLY_SLUGS}
    pdfs = [p for p in pdfs if chapter_slug_from_pdf(p) in wanted]

LOGGER.info('Run start | pdfs=%d | skipped=%d | grade_range=%d-%d | gpu_state=%s | model=%s | kind=%s', len(pdfs), len(skipped), GRADE_MIN, GRADE_MAX, gpu_summary(), MODEL_ID, MODEL_KIND)
for skipped_pdf in skipped[:10]:
    LOGGER.info('skip grade filter | pdf=%s', skipped_pdf.name)
if len(skipped) > 10:
    LOGGER.info('skip grade filter | additional=%d', len(skipped) - 10)
print(f'Found {len(pdfs)} pdf(s)')
results = []
for index, pdf_path in enumerate(pdfs, start=1):
    if MAX_CHAPTERS is not None and index > MAX_CHAPTERS:
        break
    LOGGER.info('[%d/%d] %s', index, len(pdfs), pdf_path.name)
    print(f'[{index}/{len(pdfs)}] {pdf_path.name}')
    try:
        bundle = build_chapter_bundle(pdf_path)
        if bundle.get('resume'):
            results.append({'pdf': str(pdf_path), 'status': 'cached'})
            LOGGER.info('chapter resume | pdf=%s | status=cached', pdf_path.name)
            print('  cached')
            continue
        result = generate_for_chapter(bundle)
        results.append({'pdf': str(pdf_path), **result})
        LOGGER.info('chapter done | pdf=%s | status=%s', pdf_path.name, result['status'])
        print('  generated')
    except Exception as exc:
        results.append({'pdf': str(pdf_path), 'status': 'failed', 'error': str(exc)})
        LOGGER.exception('chapter failed | pdf=%s', pdf_path.name)
        print('  failed:', exc)
        chapter_dir = output_dir_for(pdf_path)
        chapter_dir.mkdir(parents=True, exist_ok=True)
        write_artifact(chapter_dir / 'manifest.json', {
            'pdf_path': str(pdf_path),
            'pdf_hash': file_hash(pdf_path),
            'status': 'failed',
            'error': str(exc),
        })

counts = Counter(row['status'] for row in results)
LOGGER.info('Run summary | %s', dict(counts))
print(dict(counts))


In [ ]:
import shutil
archive = Path('/kaggle/working/textbook_artifacts')
zip_path = Path('/kaggle/working/textbook_artifacts.zip')
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(archive), 'zip', OUTPUT_ROOT)
print(zip_path)


## Resumability notes

- Each chapter writes its own `manifest.json`.
- Existing `summary.json`, `concepts.json`, `flashcards.json`, and `quizzes.json` are reused on rerun.
- If you export `OUTPUT_ROOT` from a previous run as a Kaggle dataset and mount it under `/kaggle/input`, the notebook will copy matching chapter folders back into the working output tree before generating. Zip exports are also unpacked automatically. If the dataset contains a `textbook_artifacts/` wrapper folder, it is detected automatically, or you can set `RESUME_SUBDIR_NAMES`.
- If Kaggle kills the session after 12 hours, rerun the notebook with the dataset mounted and it will continue from completed chapters.
- Set `ONLY_SLUGS` or `MAX_CHAPTERS` for smoke tests before a full run.
